In [3]:
import numpy as np
from astropy import units as u
# from table in Draine
alpha_b_5e3 = 4.53e-13*u.cm**3*u.s**-1
alpha_b_1e4 = 2.59e-13*u.cm**3*u.s**-1
alpha_b_2e4 = 1.43e-13*u.cm**3*u.s**-1
alpha_b_beta_1e4 = 3.03e-14*u.cm**3*u.s**-1

hb_to_paa_1e4 = 0.336
hb_to_bra_1e4 = 0.169
alpha_paa = alpha_eff(1e4, line='beta') * hb_to_paa_1e4

from pyspeckit.spectrum.models import hydrogen
wl_paa = hydrogen.wavelength['paschena']*u.um
e_paa = wl_paa.to(u.erg, u.spectral())
nu_paa = wl_paa.to(u.Hz, u.spectral())

wl_hbeta = hydrogen.wavelength['balmerb']*u.um
nu_hbeta = wl_hbeta.to(u.Hz, u.spectral())

wl_halpha = hydrogen.wavelength['balmera']*u.um
nu_halpha = wl_halpha.to(u.Hz, u.spectral())

ha_to_hb_1e4=2.86
def EMfunc(Qlyc=1e45*u.s**-1, R=0.1*u.pc, alpha_b=2e-13*u.cm**3*u.s**-1):
    return (R * (((3 * Qlyc)/(4 * np.pi * R**3 * alpha_b))**0.5)**2).to(u.cm**-6*u.pc)
def alpha_eff(T, line='beta'):
    """ H-alpha recombination coefficient.  eqn 14.8, 14.9 in draine 2001"""
    T4 = (T/(1e4*u.K)).decompose().value
    if line == 'alpha':
        return 1.17e-13 * T4**(-0.942-0.031*np.log(T4)) * u.cm**3*u.s**-1
    elif line == 'beta':
        return 3.03e-14 * T4**(-0.874-0.058*np.log(T4)) * u.cm**3*u.s**-1    
def snu_paa(Te=10000*u.K, EM=EMfunc(alpha_b=alpha_b_1e4), angular_area=4*np.pi*u.sr):
    # temperature dependence factor: jhb ~ alpha, so this accounts for the T-dependence of j
    # (which is not explicitly given in Draine)
    alpha_rel = alpha_eff(T=Te, line='beta') / alpha_b_beta_1e4
    jhb_4p = 1.24e-25 * u.erg * u.cm**3 / u.s * hb_to_paa_1e4 * alpha_rel
    flux = EM * jhb_4p
    return (flux/nu_paa).to(u.mJy)/angular_area

def snu_bra(Te=10000*u.K, EM=EMfunc(alpha_b=alpha_b_1e4), angular_area=4*np.pi*u.sr):
    # temperature dependence factor: jhb ~ alpha, so this accounts for the T-dependence of j
    # (which is not explicitly given in Draine)
    alpha_rel = alpha_eff(T=Te, line='beta') / alpha_b_beta_1e4
    jhb_4p = 1.24e-25 * u.erg * u.cm**3 / u.s * hb_to_bra_1e4 * alpha_rel 
    flux = EM * jhb_4p
    return (flux/nu_paa).to(u.mJy)/angular_area
snu_paa(Te=1e4*u.K, EM=1e6*u.cm**-6*u.pc).to(u.MJy/u.sr)

<Quantity 6.40060704 MJy / sr>

In [4]:
snu_bra(Te=1e4*u.K, EM=1e6*u.cm**-6*u.pc).to(u.MJy/u.sr)

<Quantity 3.21935295 MJy / sr>